# TP3 du cours de vision par ordinateur

In [5]:
import torch
import torchvision
import sklearn
import numpy as np
import matplotlib.pyplot as plt
import kagglehub

In [6]:
# Importer un modèle pré-entraîné (EfficientNet-B0)
model = torchvision.models.efficientnet_b0(pretrained=True, progress = True)

In [7]:
# Download latest version
path = kagglehub.dataset_download("phucthaiv02/butterfly-image-classification")
print("Path to dataset files:", path)

Path to dataset files: C:\Users\victor\.cache\kagglehub\datasets\phucthaiv02\butterfly-image-classification\versions\3


In [8]:
import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Transformations pour EfficientNet
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class ButterflyDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform
        # Encoder les labels en entiers
        self.classes = sorted(self.df["label"].unique())
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row["filename"])
        image = Image.open(img_path).convert("RGB")
        label = self.class_to_idx[row["label"]]
        if self.transform:
            image = self.transform(image)
        return image, label

full_dataset = ButterflyDataset(os.path.join(path, "Training_set.csv"), os.path.join(path, "train"), transform)
classes = full_dataset.classes

train_dataset, test_dataset = sklearn.model_selection.train_test_split(full_dataset, test_size=0.2, random_state=42)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

num_classes = len(classes)
print(f"Classes: {num_classes}")
print(f"Train: {len(train_dataset)} images | Test: {len(test_dataset)} images")

Classes: 75
Train: 5199 images | Test: 1300 images


In [9]:
# Adapter le classifier au nombre de classes du dataset
model.classifier[1] = torch.nn.Linear(in_features=1280, out_features=num_classes)
print(f"Classifier adapté pour {num_classes} classes")
print(model)

Classifier adapté pour 75 classes
EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          

In [10]:
# Finetune du modele

from torch import nn, optim
from torch.amp import autocast, GradScaler

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)
criterion = torch.nn.CrossEntropyLoss()

# Geler les couches convolutives
for module in model.features:
    for param in module.parameters():
        param.requires_grad = False

num_epochs = 10
log_interval = 50
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

scaler = GradScaler("cuda")

for epoch in range(num_epochs):
    model.train()
    for batch_idx, (data, targets) in enumerate(train_loader):
        data, targets = data.to(device), targets.to(device)

        optimizer.zero_grad()

        with autocast(device_type=device.type):
            outputs = model(data)
            loss = criterion(outputs, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        if batch_idx % log_interval == 0:
            print(f"Epoch {epoch+1}/{num_epochs} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

Epoch 1/10 | Batch 0/163 | Loss: 4.3859
Epoch 1/10 | Batch 50/163 | Loss: 3.0316
Epoch 1/10 | Batch 100/163 | Loss: 2.3022
Epoch 1/10 | Batch 150/163 | Loss: 1.3942
Epoch 2/10 | Batch 0/163 | Loss: 1.4457
Epoch 2/10 | Batch 50/163 | Loss: 1.1296
Epoch 2/10 | Batch 100/163 | Loss: 0.9402
Epoch 2/10 | Batch 150/163 | Loss: 1.0816
Epoch 3/10 | Batch 0/163 | Loss: 0.9373
Epoch 3/10 | Batch 50/163 | Loss: 1.0030
Epoch 3/10 | Batch 100/163 | Loss: 0.6389
Epoch 3/10 | Batch 150/163 | Loss: 0.6359
Epoch 4/10 | Batch 0/163 | Loss: 0.6685
Epoch 4/10 | Batch 50/163 | Loss: 0.4901
Epoch 4/10 | Batch 100/163 | Loss: 0.6120
Epoch 4/10 | Batch 150/163 | Loss: 0.6487
Epoch 5/10 | Batch 0/163 | Loss: 0.3681
Epoch 5/10 | Batch 50/163 | Loss: 0.3948
Epoch 5/10 | Batch 100/163 | Loss: 0.5142
Epoch 5/10 | Batch 150/163 | Loss: 0.4671
Epoch 6/10 | Batch 0/163 | Loss: 0.3386
Epoch 6/10 | Batch 50/163 | Loss: 0.3561
Epoch 6/10 | Batch 100/163 | Loss: 0.4508
Epoch 6/10 | Batch 150/163 | Loss: 0.4255
Epoch 7/10